In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_2.py ──
"""
Shared infrastructure for MLFP02 Exercise 2 — Parameter Estimation and Inference.

Contains: Singapore economic data loading, log-likelihood objectives for
Normal / Student-t / Laplace, profile LR CI helpers, bootstrap utilities,
plot output directory and save helper.

Technique-specific narrative (which scenario, which interpretation) belongs
in the per-technique files in ``modules/mlfp02/solutions/ex_2/``.
"""

from pathlib import Path
from typing import Callable

import numpy as np
import plotly.graph_objects as go
import polars as pl
from scipy import stats
from scipy.optimize import minimize


# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp02_ex2_mle_theory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig: go.Figure, filename: str) -> Path:
    """Write a Plotly figure to the exercise output directory."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore economic indicators
# ════════════════════════════════════════════════════════════════════════

SINGAPORE_ECON_DATASET = ("mlfp01", "economic_indicators.csv")


def load_singapore_econ() -> pl.DataFrame:
    """Load Singapore economic indicators (GDP, inflation, unemployment)."""
    loader = MLFPDataLoader()
    return loader.load(*SINGAPORE_ECON_DATASET)


def extract_series(df: pl.DataFrame, column: str) -> np.ndarray:
    """Drop nulls and return a float64 numpy array for a given column."""
    return df[column].drop_nulls().to_numpy().astype(np.float64)


def load_gdp_growth() -> np.ndarray:
    """Convenience: GDP growth (annualised quarterly %) as a 1-D numpy array."""
    return extract_series(load_singapore_econ(), "gdp_growth_pct")


# ════════════════════════════════════════════════════════════════════════
# LIKELIHOOD OBJECTIVES
# ════════════════════════════════════════════════════════════════════════
#
# All objectives are NEGATIVE log-likelihoods so they can be passed
# directly to scipy.optimize.minimize. Location/scale parameters are
# reparameterised where needed (log-sigma) to keep the optimiser in a
# valid region without explicit bounds.


def neg_log_likelihood_normal(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ N(mu, sigma^2). params = [mu, log_sigma]."""
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    if sigma <= 0:
        return np.inf
    return float(-np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma)))


def neg_log_likelihood_t(params: np.ndarray, x: np.ndarray) -> float:
    """-log L for X ~ t(df, mu, scale). params = [df, mu, scale]."""
    df, mu, scale = params
    if df <= 0 or scale <= 0:
        return np.inf
    return float(-np.sum(stats.t.logpdf(x, df=df, loc=mu, scale=scale)))


def fit_normal_mle(x: np.ndarray) -> dict:
    """Fit Normal MLE numerically via L-BFGS-B. Returns mu, sigma, loglik, result."""
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_log_likelihood_normal,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


def fit_student_t_mle(x: np.ndarray) -> dict:
    """Fit Student-t MLE via Nelder-Mead. Returns df, mu, scale, loglik."""
    x0 = np.array([5.0, float(x.mean()), float(x.std() + 1e-8)])
    result = minimize(neg_log_likelihood_t, x0, args=(x,), method="Nelder-Mead")
    return {
        "df": float(result.x[0]),
        "mu": float(result.x[1]),
        "scale": float(result.x[2]),
        "loglik": float(-result.fun),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# FISHER INFORMATION + CONFIDENCE INTERVALS
# ════════════════════════════════════════════════════════════════════════
#
# For N(mu, sigma^2):
#   Var(mu_hat)   = sigma^2 / n         =>  SE(mu_hat)  = sigma / sqrt(n)
#   Var(sigma_hat) ~ sigma^2 / (2n)     =>  SE(sigma_hat) = sigma / sqrt(2n)
#
# These come directly from inverting the Fisher information matrix at
# the MLE.


def normal_fisher_standard_errors(sigma: float, n: int) -> tuple[float, float]:
    """Return (SE_mu, SE_sigma) from Fisher information at the Normal MLE."""
    se_mu = sigma / np.sqrt(n)
    se_sigma = sigma / np.sqrt(2 * n)
    return float(se_mu), float(se_sigma)


def wald_ci(estimate: float, se: float, alpha: float = 0.05) -> tuple[float, float]:
    """Symmetric Wald CI: estimate ± z_{1-alpha/2} * SE."""
    z = float(stats.norm.ppf(1 - alpha / 2))
    return (estimate - z * se, estimate + z * se)


def profile_lr_ci_normal_mu(
    x: np.ndarray,
    mu_hat: float,
    sigma_hat: float,
    loglik_at_mle: float,
    alpha: float = 0.05,
    grid_width_in_se: float = 4.0,
    n_grid: int = 500,
) -> tuple[tuple[float, float], np.ndarray, np.ndarray]:
    """Profile likelihood 1-alpha CI for the Normal mean.

    The CI is the set of mu where 2*(loglik_at_mle - loglik(mu)) < chi^2_{1-alpha, df=1}.

    Returns (ci, mu_grid, loglik_values) so the caller can plot the profile.
    """
    n = len(x)
    se_mu = sigma_hat / np.sqrt(n)
    threshold = float(stats.chi2.ppf(1 - alpha, df=1)) / 2.0

    mu_grid = np.linspace(
        mu_hat - grid_width_in_se * se_mu,
        mu_hat + grid_width_in_se * se_mu,
        n_grid,
    )
    loglik_values = np.array(
        [-neg_log_likelihood_normal([mu, np.log(sigma_hat)], x) for mu in mu_grid]
    )
    lr_values = loglik_at_mle - loglik_values
    mask = lr_values <= threshold
    if mask.any():
        ci = (float(mu_grid[mask][0]), float(mu_grid[mask][-1]))
    else:
        # Fallback: Wald CI
        ci = wald_ci(mu_hat, se_mu, alpha)
    return ci, mu_grid, loglik_values


# ════════════════════════════════════════════════════════════════════════
# MAP — NORMAL LIKELIHOOD + NORMAL PRIOR ON THE MEAN
# ════════════════════════════════════════════════════════════════════════


def make_map_objective(
    prior_mean: float, prior_std: float
) -> Callable[[np.ndarray, np.ndarray], float]:
    """Return a neg_map_objective(params, x) closure for the given Normal prior on mu."""

    def neg_map_objective(params: np.ndarray, x: np.ndarray) -> float:
        mu, log_sigma = params
        sigma = np.exp(log_sigma)
        if sigma <= 0:
            return np.inf
        nll = -np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma))
        neg_log_prior = -stats.norm.logpdf(mu, loc=prior_mean, scale=prior_std)
        return float(nll + neg_log_prior)

    return neg_map_objective


def fit_normal_map(x: np.ndarray, prior_mean: float, prior_std: float) -> dict:
    """Fit MAP for Normal likelihood with a Normal(prior_mean, prior_std^2) prior on mu."""
    neg_map = make_map_objective(prior_mean, prior_std)
    x0 = np.array([float(x.mean()), float(np.log(x.std() + 1e-8))])
    result = minimize(
        neg_map,
        x0,
        args=(x,),
        method="L-BFGS-B",
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return {
        "mu": float(result.x[0]),
        "sigma": float(np.exp(result.x[1])),
        "converged": bool(result.success),
        "result": result,
    }


# ════════════════════════════════════════════════════════════════════════
# BOOTSTRAP UTILITIES
# ════════════════════════════════════════════════════════════════════════


def bootstrap_statistic(
    x: np.ndarray,
    statistic: Callable[[np.ndarray], float],
    n_boot: int = 10_000,
    seed: int | None = 42,
) -> np.ndarray:
    """Nonparametric bootstrap: resample x with replacement, apply statistic."""
    rng = np.random.default_rng(seed)
    n = len(x)
    out = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        out[i] = statistic(rng.choice(x, size=n, replace=True))
    return out


def bootstrap_percentile_ci(
    boot_samples: np.ndarray, alpha: float = 0.05
) -> tuple[float, float]:
    lo = float(np.percentile(boot_samples, 100 * alpha / 2))
    hi = float(np.percentile(boot_samples, 100 * (1 - alpha / 2)))
    return lo, hi


# ════════════════════════════════════════════════════════════════════════
# AIC / BIC
# ════════════════════════════════════════════════════════════════════════


def aic(k: int, loglik: float) -> float:
    return 2 * k - 2 * loglik


def bic(k: int, loglik: float, n: int) -> float:
    return k * float(np.log(n)) - 2 * loglik


# ════════════════════════════════════════════════════════════════════════
# DEFAULTS — SAMPLE SIZES, SEEDS, PRIOR VALUES
# ════════════════════════════════════════════════════════════════════════
#
# These constants are referenced by multiple technique files. Change them
# once here and every file picks up the update.

DEFAULT_SEED: int = 42
DEFAULT_N_BOOT: int = 10_000
DEFAULT_N_CLT_REPS: int = 5000

# Singapore prior on quarterly GDP growth — long-run open-economy estimate
# that we use to illustrate MAP shrinkage.
GDP_PRIOR_MEAN: float = 3.5  # %
GDP_PRIOR_STD: float = 1.5  # %




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 2.5: Model Selection and Bootstrap
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Use AIC/BIC to compare distribution families (Normal, Student-t,
#     Skew-Normal, Laplace) on the same dataset
#   - Interpret AIC vs BIC disagreement (complexity-accuracy trade-off)
#   - Implement bootstrap for non-standard statistics (median, trimmed
#     mean, IQR) where Fisher information has no formula
#   - Compare bootstrap SE to theoretical SE for the mean (validation)
#   - Visualise bootstrap distributions and confidence intervals
#
# PREREQUISITES: 04_mle_failures.py (when MLE goes wrong)
# ESTIMATED TIME: ~35 minutes
#
# TASKS (5-phase R10):
#   1. Theory — AIC/BIC derivation, bootstrap principle
#   2. Build — fit four distribution families, compute AIC/BIC
#   3. Train — bootstrap for median, trimmed mean, IQR
#   4. Visualise — bootstrap distributions and CI comparison
#   5. Apply — OCBC portfolio risk: which tail model to use?
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats



## THEORY — Model Selection and Bootstrap


In [ ]:
#
# AIC = 2k - 2 l(theta_hat)   (k = number of parameters, lower is better)
# BIC = k*log(n) - 2 l(theta_hat)  (penalises more for large n)
#
# When AIC and BIC agree, the evidence is strong. When they disagree,
# prefer BIC for prediction, AIC for explanation.
#
# BOOTSTRAP: resample with replacement, compute statistic, repeat.
# Works for ANY statistic — no analytical formulas needed.



## TASK 2 — BUILD: AIC/BIC Model Comparison


In [ ]:
econ = load_singapore_econ()
gdp_growth = extract_series(econ, "gdp_growth_pct")
n_gdp = len(gdp_growth)
rng = np.random.default_rng(seed=DEFAULT_SEED)

print(f"\n=== AIC/BIC Model Comparison ===")

fits: dict[str, dict] = {}

# TODO: Fit Normal MLE and get log-likelihood.
# Hint: fit_normal_mle(gdp_growth) returns dict with "loglik" key.
normal_result = ____
ll_normal = normal_result["loglik"]
fits["Normal"] = {"k": 2, "ll": ll_normal}

# TODO: Fit Student-t MLE and get log-likelihood.
# Hint: fit_student_t_mle(gdp_growth) returns dict with "loglik" key.
t_result = ____
ll_t = t_result["loglik"]
fits["Student-t"] = {"k": 3, "ll": ll_t}

# Skew-Normal (k=3: loc, scale, shape)
sn_params = stats.skewnorm.fit(gdp_growth)
ll_sn = float(np.sum(stats.skewnorm.logpdf(gdp_growth, *sn_params)))
fits["Skew-Normal"] = {"k": 3, "ll": ll_sn}

# Laplace (k=2: loc, scale)
lap_loc, lap_scale = stats.laplace.fit(gdp_growth)
ll_lap = float(np.sum(stats.laplace.logpdf(gdp_growth, loc=lap_loc, scale=lap_scale)))
fits["Laplace"] = {"k": 2, "ll": ll_lap}

print(f"{'Distribution':<15} {'k':>3} {'Log-lik':>12} {'AIC':>10} {'BIC':>10}")
print("-" * 55)
for name, f in fits.items():
    # TODO: Compute AIC and BIC using the shared helpers.
    # Hint: aic(k, loglik) and bic(k, loglik, n)
    f["aic"] = ____
    f["bic"] = ____
    print(
        f"{name:<15} {f['k']:>3} {f['ll']:>12.2f} {f['aic']:>10.2f} {f['bic']:>10.2f}"
    )

best_aic = min(fits.items(), key=lambda x: x[1]["aic"])
best_bic = min(fits.items(), key=lambda x: x[1]["bic"])
print(f"\nBest by AIC: {best_aic[0]} (AIC={best_aic[1]['aic']:.2f})")
print(f"Best by BIC: {best_bic[0]} (BIC={best_bic[1]['bic']:.2f})")



### Checkpoint 1


In [ ]:
assert all(
    f["aic"] is not None for f in fits.values()
), "All AIC values must be computed"
print("\n--- Checkpoint 1 passed --- AIC/BIC model selection completed\n")



## TASK 3 — TRAIN: Bootstrap for Non-Standard Statistics


In [ ]:
print(f"\n=== Bootstrap for Non-Standard Statistics ===")

# TODO: Bootstrap the median using bootstrap_statistic().
# Hint: bootstrap_statistic(data, statistic_func, n_boot=...) returns
# an array of bootstrapped values. np.median is the statistic function.
boot_medians = ____
median_ci = bootstrap_percentile_ci(boot_medians)

# TODO: Bootstrap the 10% trimmed mean.
# Hint: use lambda x: float(stats.trim_mean(x, 0.1)) as the statistic
boot_trimmed = ____
trimmed_ci = bootstrap_percentile_ci(boot_trimmed)

# Bootstrap the IQR
boot_iqrs = bootstrap_statistic(
    gdp_growth,
    lambda x: float(np.subtract(*np.percentile(x, [75, 25]))),
    n_boot=DEFAULT_N_BOOT,
)
iqr_ci = bootstrap_percentile_ci(boot_iqrs)

# Bootstrap the mean for reference
boot_means = bootstrap_statistic(
    gdp_growth,
    lambda x: float(x.mean()),
    n_boot=DEFAULT_N_BOOT,
)
mean_ci = bootstrap_percentile_ci(boot_means)

print(f"{'Statistic':<18} {'Estimate':>10} {'Boot SE':>10} {'95% CI':>25}")
print("-" * 65)
for name, est, boots in [
    ("Mean", float(gdp_growth.mean()), boot_means),
    ("Median", float(np.median(gdp_growth)), boot_medians),
    ("10% Trimmed Mean", float(stats.trim_mean(gdp_growth, 0.1)), boot_trimmed),
    ("IQR", float(np.subtract(*np.percentile(gdp_growth, [75, 25]))), boot_iqrs),
]:
    ci = bootstrap_percentile_ci(boots)
    print(
        f"{name:<18} {est:>10.4f} {boots.std():>10.4f} [{ci[0]:>10.4f}, {ci[1]:>10.4f}]"
    )

# Effect of sample size on bootstrap SE
print(f"\n--- Bootstrap SE by Sample Size ---")
for n_sub in [10, 20, 50, n_gdp]:
    sub = gdp_growth[: min(n_sub, n_gdp)]
    boot_m = bootstrap_statistic(
        sub, lambda x: float(x.mean()), n_boot=5000, seed=DEFAULT_SEED
    )
    print(
        f"  n={n_sub:>3}: Boot SE(mean) = {boot_m.std():.4f}, "
        f"Theory SE = {sub.std(ddof=1)/np.sqrt(len(sub)):.4f}"
    )



### Checkpoint 2


In [ ]:
assert (
    median_ci[0] < np.median(gdp_growth) < median_ci[1]
), "Median must be within its bootstrap CI"
assert boot_medians.std() > 0, "Bootstrap SE of median must be positive"
print("\n--- Checkpoint 2 passed --- bootstrap for non-standard statistics\n")



## TASK 4 — VISUALISE: Bootstrap Distributions


In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Bootstrap Means", "Bootstrap Medians"]
)
fig.add_trace(go.Histogram(x=boot_means, nbinsx=50, name="Means"), row=1, col=1)
fig.add_trace(go.Histogram(x=boot_medians, nbinsx=50, name="Medians"), row=1, col=2)
fig.update_layout(title="Bootstrap Distributions: Mean vs Median", height=350)
save_figure(fig, "ex2_05_bootstrap_comparison.html")
print("Saved: ex2_05_bootstrap_comparison.html")



### Checkpoint 3


In [ ]:
print("\n--- Checkpoint 3 passed --- bootstrap visualisations saved\n")



## TASK 5 — APPLY: OCBC Portfolio Risk — Which Tail Model?


In [ ]:
# OCBC manages a fixed-income portfolio. VaR at 99% confidence
# determines the capital reserve. Wrong distribution = wrong capital.

print(f"\n=== APPLY: OCBC Portfolio Risk — Distribution Choice ===")

mu_normal = normal_result["mu"]
sigma_normal = normal_result["sigma"]
t_df_fit = t_result["df"]
t_mu_fit = t_result["mu"]
t_scale_fit = t_result["scale"]

portfolio_value = 10_000  # SGD millions
print(f"Portfolio value: SGD {portfolio_value:,}M")
print(
    f"\n{'Confidence':>12} {'VaR (Normal)':>15} {'VaR (t-dist)':>15} {'Shortfall':>12}"
)
print("-" * 60)
for alpha in [0.95, 0.99, 0.995]:
    # TODO: Compute VaR under Normal and t-distribution models.
    # VaR = negative of the (1-alpha) quantile.
    # Hint: stats.norm.ppf(1-alpha, loc=..., scale=...)
    #       stats.t.ppf(1-alpha, df=..., loc=..., scale=...)
    var_normal = ____
    var_t = ____
    shortfall = (var_t - var_normal) * portfolio_value / 100
    print(
        f"{alpha*100:>10.1f}%  {var_normal:>12.3f}%  {var_t:>12.3f}%  "
        f"SGD {shortfall:>7.0f}M"
    )

if best_aic[0] == best_bic[0]:
    print(f"\nAIC and BIC agree: use {best_aic[0]} for portfolio risk.")
else:
    print(f"\nAIC prefers {best_aic[0]}, BIC prefers {best_bic[0]}.")
    print("For risk management (tail events matter), prefer the heavier-tailed model.")

print(
    "\nBottom line: choosing the wrong distribution model can leave"
    "\nhundreds of millions in unprovisioned tail risk."
)



### Checkpoint 4


In [ ]:
print("\n--- Checkpoint 4 passed --- OCBC portfolio risk application complete\n")



## REFLECTION


- AIC/BIC: penalised likelihood for distribution comparison
  - When AIC and BIC agree, the evidence is strong
  - Bootstrap for any statistic: median, trimmed mean, IQR
  - Bootstrap SE matches theoretical SE for the mean (validation)
  - Bootstrap is the ONLY practical option for non-standard statistics
  - Real-world impact: VaR capital reserves under different models

  SUMMARY — Exercise 2 Complete:
    2.1  CLT and Sampling — why x-bar is Normal regardless of population
    2.2  MLE and Fisher — optimise log-likelihood, standard errors, CIs
    2.3  MAP Estimation — MLE + prior = Bayesian regularisation
    2.4  MLE Failures — small n, multimodal, misspecification
    2.5  Model Selection — AIC/BIC + bootstrap for robust inference

  NEXT: In Exercise 3, you'll move from estimation to decision-making.
  You'll formulate null and alternative hypotheses, run power analysis,
  apply multiple testing corrections (Bonferroni and BH-FDR),
  implement a permutation test, and simulate false discovery rates
  — all on A/B test data.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print("--- Exercise 2.5 complete --- Model Selection and Bootstrap")

